In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    get_stimulus_onset_samples,
    DEFAULT_STIMULUS_LABEL,
)
from src.filtering.dataset_filter import DatasetFilter  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
    PreprocessedDataVariants,
    SingleDataMetadata,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# Stimulus Alignment — ASSR

Aligns stimulus onsets (annotated `fam+`) across the participants of one group so
that stimulus *k* falls at the **same sample index** in every recording, enabling
sample-locked cross-participant analysis.

The workflow is two-stage (see `src/preprocessing/stimulus_alignment.py`):

1. **Coarse crop** (during preprocessing): each recording is trimmed by up to
   `trim_sec` from each end, never closer than `min_keep_sec` to the first/last
   onset. Continuous (no splicing) so filtering/ICA are clean.
2. **Fine alignment** (here): every inter-stimulus interval is trimmed from the
   front to the group-wide minimum (preserving `keep_tail` of data before each
   onset), and the EEG is spliced. Edges keep the shortest remaining lead-in/out.

This notebook runs the fine alignment on the preprocessed (`after_ica`) data and
inspects the result.

> **Requires preprocessing to have run** for the selected experiment
> (`python scripts/run_preprocessing.py --experiment assr --raw_processing`).

## Configuration

In [ ]:
# ── Group to align ────────────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR  # ASSR has a single (placeholder) music type
EXCLUSION_CATEGORIES: list[ExclusionCategories] = [ExclusionCategories.WRONG_CONDITION, ExclusionCategories.ARTIFACTS]  # e.g. [ExclusionCategories.ARTIFACTS]

# ── Alignment parameters ──────────────────────────────────────────────────────
STIMULUS_LABEL = DEFAULT_STIMULUS_LABEL  # "fam+"
KEEP_TAIL_SEC = 0.1  # continuous data kept immediately before each onset
PRE_WINDOW_SEC = None  # None -> shortest remaining lead-in across the group
POST_WINDOW_SEC = None  # None -> shortest remaining lead-out across the group
DATA_STAGE = PreprocessedDataVariants.RAW_AFTER_ICA

# ── Reuse / compute ───────────────────────────────────────────────────────────
# When True and a complete set of `cropped` recordings already exists for this
# group, load them directly instead of re-running the (expensive) alignment —
# mirrors REUSE_WAVELETS in the 03/04/05 wavelet notebooks. The drift / interval
# inspection cells need the *original* (pre-alignment) data, so they are skipped
# on the cached path; the alignment-quality checks below run either way.
REUSE_ALIGNMENT = True
# When alignment is (re)computed, persist the spliced recordings to the `cropped`
# stage so subsequent runs can reuse them.
SAVE_ALIGNMENT = True

# ── ASSR response inspection ──────────────────────────────────────────────────
EPOCH_TMIN, EPOCH_TMAX = -0.1, 1.2  # window around each fam+ onset (s)
ASSR_FREQ = 40.0  # expected steady-state frequency (Hz)

# ── Plot saving ───────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "stimulus_alignment"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Group: {EXPERIMENT.value} / {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Reuse cached alignment: {REUSE_ALIGNMENT}")
print(f"Plots -> {PLOTS_DIR}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
print(f"Total recordings parsed: {len(dataset_handler.dataset_metadata)}")
dataset_handler.dataset_metadata.head()

## Run (or Load) Alignment

Loads the `after_ica` recordings for the selected group, trims/splices them, and
returns the per-group metadata, the aligned recordings, and the fitted
`StimulusAligner` (which holds the alignment plan).

If `REUSE_ALIGNMENT` is set and a complete set of `cropped` recordings already
exists for this group, those are loaded directly and the alignment is **not**
recomputed (`aligner` is then `None`). Otherwise the alignment runs and — when
`SAVE_ALIGNMENT` is set — the spliced recordings are written to the `cropped`
stage for reuse (the same artefacts produced by
`scripts/run_stimulus_alignment.py`).

In [ ]:
# Metadata for the group is needed on both paths (cache load and recompute).
filtered_df = DatasetFilter.filter_dataset_by_all_categories(
    dataset_handler.dataset_metadata,
    dataset_handler.excluded_participants_metadata,
    [MUSIC_TYPE],
    [CONDITION],
    EXCLUSION_CATEGORIES,
)

# Is a complete set of cropped recordings already on disk for this group?
cropped_paths = [
    dataset_handler.get_preprocessing_results_path(
        fname.split(".")[0], PreprocessedDataVariants.RAW_CROPPED
    )
    for fname in filtered_df[SingleDataMetadata.FILENAME]
]
cache_complete = len(cropped_paths) > 0 and all(p.exists() for p in cropped_paths)

if REUSE_ALIGNMENT and cache_complete:
    # ── Cached path: load the spliced recordings directly. ────────────────────
    aligned = [
        dataset_handler.load_data_file(
            fname,
            is_processed=True,
            processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
            preload=True,
        )
        for fname in filtered_df[SingleDataMetadata.FILENAME]
    ]
    aligner = None  # no plan object on the cached path
    print(f"Loaded {len(aligned)} cached cropped recordings (alignment NOT recomputed).")
else:
    # ── Compute path: run the alignment (and optionally persist it). ──────────
    if REUSE_ALIGNMENT and not cache_complete:
        missing = [p.name for p in cropped_paths if not p.exists()]
        print(f"Cropped cache incomplete ({len(missing)} missing) -> recomputing.")
    filtered_df, aligned, aligner = dataset_handler.align_stimuli_by_annotations(
        music_type=MUSIC_TYPE,
        condition_type=CONDITION,
        exclusion_categories=EXCLUSION_CATEGORIES,
        stimulus_label=STIMULUS_LABEL,
        keep_tail_sec=KEEP_TAIL_SEC,
        pre_window_sec=PRE_WINDOW_SEC,
        post_window_sec=POST_WINDOW_SEC,
        data_type_to_load=DATA_STAGE,
    )
    if SAVE_ALIGNMENT:
        for (_, row), cropped in zip(filtered_df.iterrows(), aligned):
            fname = row[SingleDataMetadata.FILENAME]
            assert cropped.n_times == aligner.total_length, (
                f"Aligned signal of file {fname} has length {cropped.n_times}, "
                f"expected {aligner.total_length}!"
            )
            dataset_handler.save_data_file(
                cropped, fname.split(".")[0], PreprocessedDataVariants.RAW_CROPPED
            )
        print(f"Saved {len(aligned)} cropped recordings to the cache.")

# ── Common handles derived from the aligned recordings (both paths). ──────────
labels = filtered_df[SingleDataMetadata.PARTICIPANT_ID].astype(str).tolist()
sfreq = aligned[0].info["sfreq"]
# Onset positions in each aligned recording (a recording may keep a trailing
# extra onset beyond the common count, so take the group minimum).
aligned_onsets_per_rec = [
    get_stimulus_onset_samples(a, STIMULUS_LABEL) for a in aligned
]
common_count = min(len(o) for o in aligned_onsets_per_rec)

print(f"Recordings in group   : {len(aligned)}")
print(f"Common stimulus count : {common_count}")
print(f"Aligned length        : {aligned[0].n_times / sfreq:.2f} s "
      f"({aligned[0].n_times} samples)")
print(f"All aligned lengths equal: {len({a.n_times for a in aligned}) == 1}")

## Per-Recording Summary

In [ ]:
if aligner is not None:
    # Full summary including pre-alignment lengths / counts (compute path only).
    summary = pd.DataFrame(
        {
            "participant": labels,
            "condition": filtered_df[SingleDataMetadata.CONDITION]
            .map(lambda c: c.value)
            .tolist(),
            "orig_fam_count": aligner.original_counts,
            "loaded_length_s": [n / sfreq for n in aligner.recording_lengths],
            "aligned_length_s": [a.n_times / sfreq for a in aligned],
            "removed_s": [
                (n - aligner.total_length) / sfreq for n in aligner.recording_lengths
            ],
        }
    )
else:
    # Cached path: only the aligned recordings are available.
    summary = pd.DataFrame(
        {
            "participant": labels,
            "condition": filtered_df[SingleDataMetadata.CONDITION]
            .map(lambda c: c.value)
            .tolist(),
            "aligned_fam_count": [len(o) for o in aligned_onsets_per_rec],
            "aligned_length_s": [a.n_times / sfreq for a in aligned],
        }
    )
print(f"All aligned lengths equal: {len({a.n_times for a in aligned}) == 1}")
summary

## Inter-Stimulus Intervals

Original inter-onset intervals per recording (truncated to the common count) and
the per-interval **target** (group minimum) the alignment trims everything down to.

> Requires the alignment plan (`aligner`), so this is **skipped when loading from
> the `cropped` cache** — set `REUSE_ALIGNMENT = False` to recompute and see it.

In [ ]:
if aligner is None:
    print("Skipped: needs the alignment plan (loaded from cache). "
          "Set REUSE_ALIGNMENT = False to recompute.")
else:
    # Original onset gaps (seconds), per recording, over the common stimulus count.
    orig_onsets = [o[:common_count] for o in aligner.onset_samples]
    gaps = [np.diff(o) / sfreq for o in orig_onsets]
    targets = aligner.interval_targets / sfreq

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    # Left: gap per interval index, one line per recording, plus the target.
    for label, g in zip(labels, gaps):
        axes[0].plot(g, alpha=0.4, lw=0.8)
    axes[0].plot(targets, color="black", lw=2.0, label="target (group min)")
    axes[0].set_title("Inter-stimulus interval per index")
    axes[0].set_xlabel("Stimulus interval index")
    axes[0].set_ylabel("Interval (s)")
    axes[0].legend()

    # Right: distribution of all original gaps vs targets.
    axes[1].hist(
        np.concatenate(gaps), bins=60, alpha=0.6, label="original gaps (all recordings)"
    )
    axes[1].hist(targets, bins=60, alpha=0.6, label="targets (per-interval min)")
    axes[1].set_title("Interval distribution")
    axes[1].set_xlabel("Interval (s)")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "inter_stimulus_intervals.png", dpi=150)
    plt.show()

## Onset Alignment (before vs after)

Onset time relative to the first onset, per recording. Before alignment the
recordings drift apart as intervals accumulate; after alignment every recording
shares the same onset positions (all lines collapse onto one).

> Requires the alignment plan (`aligner`), so this is **skipped when loading from
> the `cropped` cache**. The cached path verifies the *after* state directly in
> the alignment-quality checks below.

In [ ]:
if aligner is None:
    print("Skipped: needs the alignment plan (loaded from cache). "
          "Set REUSE_ALIGNMENT = False to recompute.")
else:
    idx = np.arange(common_count)
    before = [(o - o[0]) / sfreq for o in orig_onsets]  # per recording
    after = (aligner.aligned_onset_samples - aligner.aligned_onset_samples[0]) / sfreq

    fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
    for label, b in zip(labels, before):
        axes[0].plot(idx, b, alpha=0.5, lw=0.8)
    axes[0].set_title("Before: onset time (rel. to first)")
    axes[0].set_xlabel("Stimulus index")
    axes[0].set_ylabel("Time since first onset (s)")

    for label in labels:  # all identical -> draw once per recording to show overlap
        axes[1].plot(idx, after, alpha=0.5, lw=0.8)
    axes[1].set_title("After: onset time (rel. to first) — all aligned")
    axes[1].set_xlabel("Stimulus index")

    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "onset_alignment_before_after.png", dpi=150)
    plt.show()

    # Cumulative drift across recordings (spread at the last stimulus).
    last_spread_before = max(b[-1] for b in before) - min(b[-1] for b in before)
    print(f"Spread at last stimulus — before: {last_spread_before:.3f} s, after: 0.000 s")

## Verify Alignment

In [ ]:
# Compare the first `common_count` onsets: a recording may keep a trailing extra
# onset (beyond the common count) within its post-window, which is expected.
# Works on the loaded aligned recordings, so it runs on both the cache and the
# compute paths.
aligned_onsets = [o[:common_count] for o in aligned_onsets_per_rec]
identical = all(
    np.array_equal(aligned_onsets[0], other) for other in aligned_onsets[1:]
)
extra = [len(o) - common_count for o in aligned_onsets_per_rec]
print(f"First {common_count} onset positions identical across recordings: {identical}")
print(f"Trailing extra onsets kept per recording (beyond common count): {extra}")
print(f"First / last aligned onset (samples): "
      f"{aligned_onsets[0][0]} / {aligned_onsets[0][-1]}")

# When the plan is available, also confirm the realised onsets match it exactly.
if aligner is not None:
    print(
        "Matches the planned positions: "
        f"{np.array_equal(aligned_onsets[0], aligner.aligned_onset_samples)}"
    )

## Alignment Quality Checks

Two checks that validate the alignment **directly from the loaded recordings**, so
they run whether the data was just aligned or loaded from the `cropped` cache:

1. **Stimulus overlap** — every recording's `fam+` onsets should land on the
   *same* sample indices. Plotted as an onset raster (perfectly vertical columns
   = perfect overlap) plus the per-onset spread across recordings (should be 0).
2. **Inter-subject cross-correlation** — the stimulus-locked component is shared
   across participants, so each subject's (channel-averaged) signal cross-correlated
   against the group template should peak at **lag 0**. A best-lag histogram tightly
   centred on zero confirms the recordings are sample-locked; non-zero lags would
   reveal residual misalignment.

In [ ]:
# ── Stimulus overlap (onset raster) ───────────────────────────────────────────
# Onsets truncated to the common count; rows = recordings. If aligned, every
# column is a single vertical line (all recordings share the onset sample).
onset_matrix = np.stack(aligned_onsets)  # (n_recordings, common_count)
onset_spread = onset_matrix.max(axis=0) - onset_matrix.min(axis=0)  # per onset index

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for row, onsets in enumerate(onset_matrix):
    axes[0].scatter(onsets / sfreq, np.full(common_count, row), s=6, alpha=0.5)
axes[0].set_title("Onset raster (rows = recordings)")
axes[0].set_xlabel("Onset time (s)")
axes[0].set_ylabel("Recording index")

axes[1].plot(onset_spread, marker="o", ms=3, lw=0.8)
axes[1].set_title("Per-onset spread across recordings (max − min)")
axes[1].set_xlabel("Stimulus index")
axes[1].set_ylabel("Spread (samples)")
axes[1].set_ylim(bottom=-0.5)

plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "stimulus_overlap_raster.png", dpi=150)
plt.show()

max_spread = int(onset_spread.max())
print(f"Max onset spread across recordings: {max_spread} samples "
      f"({max_spread / sfreq * 1000:.2f} ms) — expect 0 for perfect overlap.")

In [ ]:
# ── Inter-subject cross-correlation / best-lag check ──────────────────────────
# Per subject: channel-averaged signal (keeps the cross-subject common-mode, i.e.
# the stimulus-locked component), z-scored. Cross-correlate each against the group
# template and find the lag of maximum correlation. Well-aligned recordings peak
# at lag 0.
from scipy.signal import correlate, correlation_lags  # noqa: E402

MAX_LAG_SEC = 0.5  # only search lags within +/- this window

sigs = []
for a in aligned:
    d = a.get_data(picks="eeg").mean(axis=0)  # (n_times,)
    d = (d - d.mean()) / (d.std() + 1e-12)
    sigs.append(d)
sigs = np.stack(sigs)  # (n_subjects, n_times)
template = sigs.mean(axis=0)
template = (template - template.mean()) / (template.std() + 1e-12)

n_times = sigs.shape[1]
lags = correlation_lags(n_times, n_times, mode="full")
max_lag = int(round(MAX_LAG_SEC * sfreq))
window = np.abs(lags) <= max_lag
lags_win = lags[window]

best_lags = np.empty(len(sigs), dtype=int)
zero_lag_corr = np.empty(len(sigs))
for i, s in enumerate(sigs):
    cc = correlate(s, template, mode="full", method="fft")[window]
    cc = cc / n_times  # normalise to ~correlation coefficient (z-scored inputs)
    best_lags[i] = lags_win[np.argmax(cc)]
    zero_lag_corr[i] = cc[lags_win == 0][0]

print(f"Best-lag range: [{best_lags.min()}, {best_lags.max()}] samples "
      f"(+/- {max_lag} searched); median |lag| = {np.median(np.abs(best_lags)):.1f}.")
print(f"Recordings peaking exactly at lag 0: {(best_lags == 0).sum()} / {len(best_lags)}")
print(f"Mean zero-lag correlation to template: {zero_lag_corr.mean():.3f}")

### Per-Participant Signal Overlay (Global Field Power)

Per-participant signal over time, overlaid on a common time axis — one line per
participant, one subplot per condition (Placebo and Psilocybin).

Following the project-standard format, each channel is first **z-scored in time per
participant**, then the per-participant line is the **Global Field Power** of those
z-scored channels (std across channels per timepoint), so the lines are on a
comparable scale (no further normalization).

Two variants are plotted (**without** vs **with** the excluded participants), and in
*both* the cell independently **suggests participants for exclusion** as amplitude
outliers (robust modified z-score on peak GFP > `OUTLIER_Z_THRESH`). Each highlighted
line is labelled with its **exclusion reason** from the metadata (artifacts,
wrong_condition, …) and/or its suggested peak.

Highlighted lines are coloured by **class (hue family)** and given a distinct
**shade + line style per individual** within that class, so you can read the class
*and* tell the individuals apart (the legend names each):

- 🟢 **green family** — excluded **and** an amplitude outlier → exclusion validated.
- 🔴 **red family** — an amplitude outlier but **not** excluded → possible *missed* exclusion.
- 🟠 **orange family** — excluded but **not** an amplitude outlier → excluded for another
  reason (shown in the label, e.g. `wrong_condition`); expected, not amplitude-driven.
- grey — kept, unremarkable.

Each group reuses its aligned `cropped` recordings when present; if the cache is
incomplete it re-runs the alignment in memory for that group, without writing to disk.

In [ ]:
# ── Per-participant Global Field Power (z-scored channels) ────────────────────
# Each channel is z-scored in time per participant (project-standard format), then
# GFP = std across the z-scored channels per timepoint gives one line per
# participant. Two variants per run (WITHOUT vs WITH excluded participants); both
# flag amplitude outliers as exclusion suggestions and label each highlighted line
# with its exclusion reason. Highlighted lines are coloured by CLASS (hue family)
# with a distinct SHADE + line style per individual so both are readable:
#   green family  = excluded & outlier (validated)
#   red family    = outlier & NOT excluded (possible miss)
#   orange family = excluded for a non-amplitude reason (see label)
CONDITIONS_TO_PLOT = [ConditionVariants.PLACEBO, ConditionVariants.PSILOCYBIN]
OUTLIER_Z_THRESH = 3.5  # modified z-score cutoff on peak GFP

CLASS_CMAPS = {"validated": plt.cm.Greens, "missed": plt.cm.Reds, "other": plt.cm.Oranges}
CLASS_LINESTYLES = ["-", "--", "-.", ":"]


def _channel_gfp(raw):
    """GFP after z-scoring each channel in time (per participant)."""
    data = raw.get_data(picks="eeg")  # (n_channels, n_times)
    z = (data - data.mean(axis=1, keepdims=True)) / (
        data.std(axis=1, keepdims=True) + 1e-12
    )
    return z.std(axis=0)  # (n_times,) — std across the z-scored channels


def _group_df(condition, exclusion_categories):
    return DatasetFilter.filter_dataset_by_all_categories(
        dataset_handler.dataset_metadata,
        dataset_handler.excluded_participants_metadata,
        [MUSIC_TYPE],
        [condition],
        exclusion_categories,
    )


def exclusion_reasons(music_type, condition):
    """Map filename / participant -> exclusion reason(s) from the metadata.

    A blank condition or music type in the metadata acts as a wildcard (applies to
    all). Returns (by_filename, by_participant) dicts of reason strings.
    """
    em = dataset_handler.excluded_participants_metadata
    cond = em[SingleDataMetadata.CONDITION.value].fillna("").str.strip()
    music = em[SingleDataMetadata.MUSIC_TYPE.value].fillna("").str.strip()
    keep = ((cond == "") | (cond == condition.value)) & (
        (music == "") | (music == music_type.value)
    )
    by_file, by_pid = {}, {}
    for _, r in em[keep].iterrows():
        reason = str(r[SingleDataMetadata.EXCLUSION_EXPLANATION.value]).strip()
        fname = str(r[SingleDataMetadata.FILENAME.value]).strip()
        pid = str(r[SingleDataMetadata.PARTICIPANT_ID.value]).strip()
        if fname and fname.lower() != "nan":
            by_file.setdefault(fname, set()).add(reason)
        else:
            by_pid.setdefault(pid, set()).add(reason)
    return by_file, by_pid


def reason_for(filename, pid, by_file, by_pid):
    reasons = by_file.get(filename, set()) | by_pid.get(pid, set())
    return "/".join(sorted(reasons))


def aligned_gfp(condition, exclusion_categories):
    """Return (labels, filenames, gfp_list, sfreq) for a group.

    Reuses the `cropped` cache when complete (GFP streamed per recording to keep
    memory low); otherwise re-runs the alignment in memory (no disk writes).
    """
    fdf = _group_df(condition, exclusion_categories)
    filenames = fdf[SingleDataMetadata.FILENAME].tolist()
    paths = [
        dataset_handler.get_preprocessing_results_path(
            f.split(".")[0], PreprocessedDataVariants.RAW_CROPPED
        )
        for f in filenames
    ]
    if REUSE_ALIGNMENT and len(paths) > 0 and all(p.exists() for p in paths):
        labels = fdf[SingleDataMetadata.PARTICIPANT_ID].astype(str).tolist()
        gfp, sfreq = [], None
        for f in filenames:
            raw = dataset_handler.load_data_file(
                f,
                is_processed=True,
                processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
                preload=True,
            )
            sfreq = raw.info["sfreq"]
            gfp.append(_channel_gfp(raw))
            del raw
        return labels, filenames, gfp, sfreq
    # Recompute in memory (e.g. excluded recordings have no cached crop).
    fdf, aligned, _ = dataset_handler.align_stimuli_by_annotations(
        music_type=MUSIC_TYPE,
        condition_type=condition,
        exclusion_categories=exclusion_categories,
        stimulus_label=STIMULUS_LABEL,
        keep_tail_sec=KEEP_TAIL_SEC,
        pre_window_sec=PRE_WINDOW_SEC,
        post_window_sec=POST_WINDOW_SEC,
        data_type_to_load=DATA_STAGE,
    )
    labels = fdf[SingleDataMetadata.PARTICIPANT_ID].astype(str).tolist()
    filenames = fdf[SingleDataMetadata.FILENAME].tolist()
    gfp = [_channel_gfp(a) for a in aligned]
    return labels, filenames, gfp, aligned[0].info["sfreq"]


def amplitude_outliers(peak_gfp, thresh):
    """Indices whose peak GFP is a high outlier by robust modified z-score."""
    med = np.median(peak_gfp)
    mad = np.median(np.abs(peak_gfp - med))
    if mad == 0:  # degenerate spread -> fall back to std
        mod_z = (peak_gfp - peak_gfp.mean()) / (peak_gfp.std() + 1e-12)
    else:
        mod_z = 0.6745 * (peak_gfp - med) / mad
    return set(np.where(mod_z > thresh)[0].tolist())


def _styles_by_class(class_of):
    """Per-index (color, linestyle): hue family by class, distinct shade per member."""
    members = {}
    for i, cls in class_of.items():
        members.setdefault(cls, []).append(i)
    style = {}
    for cls, idxs in members.items():
        cmap = CLASS_CMAPS[cls]
        for k, i in enumerate(idxs):
            frac = 0.7 if len(idxs) == 1 else 0.55 + 0.4 * k / (len(idxs) - 1)
            style[i] = (cmap(frac), CLASS_LINESTYLES[k % len(CLASS_LINESTYLES)])
    return style


def plot_gfp_variant(drop_excluded):
    """One figure, one subplot per condition. ``drop_excluded`` toggles the variant."""
    variant = "without excluded" if drop_excluded else "with excluded"
    cats = EXCLUSION_CATEGORIES if drop_excluded else []
    fig, axes = plt.subplots(
        len(CONDITIONS_TO_PLOT),
        1,
        figsize=(15, 5 * len(CONDITIONS_TO_PLOT)),
        squeeze=False,
    )
    for ax, condition in zip(axes[:, 0], CONDITIONS_TO_PLOT):
        labels, filenames, gfp, sfreq = aligned_gfp(condition, cats)
        peak_gfp = np.array([g.max() for g in gfp])
        times = np.arange(len(gfp[0])) / sfreq

        included = set(
            _group_df(condition, EXCLUSION_CATEGORIES)[SingleDataMetadata.FILENAME]
        )
        by_file, by_pid = exclusion_reasons(MUSIC_TYPE, condition)
        outliers = amplitude_outliers(peak_gfp, OUTLIER_Z_THRESH)
        excluded_now = [filenames[i] not in included for i in range(len(gfp))]

        # Classify highlighted recordings, then assign per-individual styles.
        class_of = {}
        for i in range(len(gfp)):
            is_out = i in outliers
            if is_out and excluded_now[i]:
                class_of[i] = "validated"
            elif is_out:
                class_of[i] = "missed"
            elif excluded_now[i]:
                class_of[i] = "other"
        style = _styles_by_class(class_of)

        for i in range(len(gfp)):
            if i not in style:
                ax.plot(times, gfp[i], lw=0.6, alpha=0.35, color="0.6")

        excl_report, suggest_report, missed_report = [], [], []
        for i in sorted(style):
            is_out = i in outliers
            reason = reason_for(filenames[i], labels[i], by_file, by_pid)
            parts = []
            if excluded_now[i]:
                parts.append(f"excluded: {reason or 'yes'}")
                excl_report.append(f"{labels[i]}({reason or '?'})")
            elif reason:
                parts.append(f"flagged: {reason}")
            if is_out:
                parts.append(f"suggest peak {peak_gfp[i]:.2f}")
                suggest_report.append(labels[i])
                if not excluded_now[i]:
                    missed_report.append(labels[i])
            color, ls = style[i]
            ax.plot(times, gfp[i], lw=1.8, alpha=0.95, color=color, ls=ls, zorder=5,
                    label=f"{labels[i]} — {', '.join(parts)}")

        ax.set_title(
            f"Per-participant GFP (z-scored channels, {variant}) — {condition.value} / "
            f"{MUSIC_TYPE.value} (n={len(gfp)})"
        )
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("GFP (z-scored channels)")
        if style:
            ax.legend(loc="upper right", ncol=2, fontsize=7, framealpha=0.6)
        print(f"[{variant}] {condition.value}: excluded={excl_report or 'none'} | "
              f"suggested(outliers)={suggest_report or 'none'} | "
              f"outlier-not-excluded={missed_report or 'none'}")

    plt.tight_layout()
    if SAVE_PLOTS:
        suffix = "without_excluded" if drop_excluded else "with_excluded"
        fig.savefig(PLOTS_DIR / f"participant_gfp_overlay_{suffix}.png", dpi=150)
    plt.show()


plot_gfp_variant(drop_excluded=True)  # without excluded participants
plot_gfp_variant(drop_excluded=False)  # with excluded participants

## ASSR Response

Epoch the aligned recordings around each `fam+` onset and average. The auditory
steady-state response should show power concentrated near the stimulation frequency
(~40 Hz). Seam annotations from splicing are ignored (`reject_by_annotation=False`)
since artifacts were already handled in preprocessing.

In [ ]:
# Epoch each recording around its fam+ onsets, then grand-average the per-recording
# evokeds (robust to per-subject info differences, and the standard cross-subject
# approach). `fam+` is regex-escaped for events_from_annotations.
label_regexp = rf"^{STIMULUS_LABEL.replace('+', chr(92) + '+')}$"

epochs_list = []
evokeds = []
for raw in aligned:
    events, _ = mne.events_from_annotations(raw, regexp=label_regexp)
    if len(events) == 0:
        continue
    ep = mne.Epochs(
        raw,
        events,
        tmin=EPOCH_TMIN,
        tmax=EPOCH_TMAX,
        baseline=(EPOCH_TMIN, 0.0),
        picks="eeg",
        reject_by_annotation=False,
        preload=True,
    )
    epochs_list.append(ep)
    evokeds.append(ep.average())

print(f"Recordings epoched: {len(evokeds)} | epochs each: {[len(e) for e in epochs_list]}")
grand = mne.grand_average(evokeds)

fig = grand.plot(spatial_colors=True, show=False)
fig.suptitle("Grand-average ASSR evoked response")
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_evoked.png", dpi=150)
plt.show()

### Response Spectrum

In [ ]:
# Per-recording epoch-mean spectrum (over epochs and channels), then averaged
# across recordings; expect a peak near ASSR_FREQ.
freqs = None
spectra = []
for ep in epochs_list:
    psd = ep.compute_psd(fmin=2.0, fmax=80.0)
    psds, freqs = psd.get_data(return_freqs=True)
    spectra.append(psds.mean(axis=(0, 1)))
mean_psd = np.mean(spectra, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freqs, 10 * np.log10(mean_psd))
ax.axvline(ASSR_FREQ, color="red", ls="--", lw=1, label=f"{ASSR_FREQ:.0f} Hz")
ax.set_title("Epoch-average power spectrum (across channels)")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power (dB)")
ax.legend()
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_spectrum.png", dpi=150)
plt.show()

### Response Time-Frequency

Time-frequency representation of the evoked (phase-locked) response. Morlet
wavelets over 2–80 Hz, computed per recording then grand-averaged (consistent
with the evoked grand-average above) and baseline-corrected (log-ratio) against
the pre-stimulus window. The ASSR should appear as a sustained band of power at
`ASSR_FREQ` (~40 Hz) for the duration of the stimulus.

In [ ]:
# Per-recording evoked TFR (Morlet), grand-averaged across recordings. n_cycles
# scales with frequency for a sensible time/frequency trade-off; the lower bound
# and cycle count are kept small enough that the wavelets fit the short epoch.
tfr_freqs = np.arange(1.0, 60.0, 1.0)
n_cycles = tfr_freqs / 4.0

evoked_tfrs = [
    ev.compute_tfr(
        method="morlet",
        freqs=tfr_freqs,
        n_cycles=n_cycles,
        verbose=False,
    )
    for ev in evokeds
]
grand_tfr = mne.grand_average(evoked_tfrs)
grand_tfr.apply_baseline(baseline=(EPOCH_TMIN, 0.0), mode="logratio")

# Average across channels into a single time-frequency map.
fig = grand_tfr.plot(
    combine="mean",
    title="Grand-average evoked TFR (across channels)",
    show=False,
)[0]
fig.axes[0].axhline(ASSR_FREQ, color="red", ls="--", lw=1)
# plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "assr_evoked_tfr.png", dpi=150)
plt.show()